In [ ]:
import torch
import numpy as np
import nvdiffrast.torch as dr
import imageio

In [18]:
DEVICE = 'cuda'
RESOLUTION = [512, 512]
DTYPE_float = torch.float32
DTYPE_int = torch.int32

In [154]:
def render_target(filename: str):
    ctx = dr.RasterizeCudaContext()

    pos = torch.tensor([
        [0.0, 0.5, 0.0, 1.0],
        [-0.5, -0.5, 0.0, 1.0],
        [0.5, -0.5, 0.0, 1.0]
    ], dtype=DTYPE_float, device=DEVICE).unsqueeze(0)

    tri = torch.tensor([[0, 1, 2]], dtype=DTYPE_int, device=DEVICE)

    face_colors = torch.tensor([0.18, 0.31, 0.67], dtype=DTYPE_float, device=DEVICE)

    rast_out, _ = dr.rasterize(ctx, pos, tri, resolution=RESOLUTION)

    tri_ids = rast_out[..., 3].long()

    background = torch.tensor([0.0, 0.0, 0.0], device=DEVICE)
    palette = torch.stack([background, face_colors])

    hard_image = palette[tri_ids]

    image = dr.antialias(hard_image, rast_out, pos, tri)

    img = image[0].detach().cpu().numpy()
    img = np.flip(img, axis=0)
    imageio.imwrite(filename, (img * 255).astype(np.uint8))

    return image, img

In [146]:
out = render_target('target.png')

In [147]:
import torch.nn as nn
import torchvision.transforms.functional as TF

In [211]:
class Scene(nn.Module):

    def __init__(self):
        super().__init__()

        self.ctx = dr.RasterizeCudaContext()
        self.resolution = [512, 512]

        self.pos = nn.Parameter(torch.randn((3, 3), dtype=torch.float32), requires_grad=True)
        self.color = nn.Parameter(torch.randn((1, 3), dtype=torch.float32), requires_grad=True)

        self.register_buffer('background', torch.tensor([[0.0, 0.0, 0.0]], dtype=torch.float32))
        self.register_buffer('tri', torch.tensor([[0, 1, 2]], dtype=torch.int32))

    def forward(self):

        pos = torch.cat([self.pos, torch.ones(self.pos.shape[0], 1, device=self.pos.device)], dim=1).unsqueeze(0)

        rast_out, _ = dr.rasterize(self.ctx, pos, self.tri, self.resolution)
        tri_ids = rast_out[..., 3].long()

        palette = torch.cat([self.background, self.color], dim=0)
        hard_image = palette[tri_ids]

        final_image = dr.antialias(hard_image, rast_out, pos, self.tri)
        return final_image

In [212]:
scene = Scene().cuda()
optimizer = torch.optim.AdamW(scene.parameters(), lr=0.02)
loss_fn = torch.nn.MSELoss()
target, target_np = render_target('target.png')

In [213]:
iteration = 300
frames = []
i = 0
loss = 1
while i < iteration and loss > 0.0000009:

    out_img = scene()
    out_blur = TF.gaussian_blur(out_img.permute(0, 3, 1, 2), kernel_size=21, sigma=5.0)
    target_blur = TF.gaussian_blur(target.permute(0, 3, 1, 2), kernel_size=21, sigma=5.0)


    loss_sharp = loss_fn(out_img, target)
    loss_blur = loss_fn(out_blur, target_blur)

    width = scene.pos[:, 0].max() - scene.pos[:, 0].min()
    height = scene.pos[:, 1].max() - scene.pos[:, 1].min()
    area_approx = width * height

    surface_loss = torch.sigmoid((0.8 - area_approx) * 80)
    loss = loss_sharp + (loss_blur * 2.0) + surface_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        scene.pos.clamp_(-1.0, 1.0)
        scene.color.clamp_(0.0, 1.0)

    if i % 10 == 0 or loss <= 0.0000001:
        print(f"Iter {i}: Loss = {loss.item():.6f}")


    current_np = out_img[0].detach().cpu().numpy()
    current_np = np.flip(current_np, axis=0)

    blended = (target_np * 0.3) + (current_np * 0.9)
    blended = np.clip(blended, 0, 1)

    frame = (np.flip(blended, axis=0) * 255).astype(np.uint8)
    frames.append(frame)

    i += 1


imageio.mimsave('training_evolution.gif', frames, fps=30)

print("\n--- RESULTS ---")
print("Position :\n", scene.pos.detach().cpu().numpy())
print("Color :", scene.color.detach().cpu().numpy())

Iter 0: Loss = 0.319063
Iter 10: Loss = 0.059226
Iter 20: Loss = 0.055373
Iter 30: Loss = 0.050211
Iter 40: Loss = 0.041826
Iter 50: Loss = 0.034944
Iter 60: Loss = 0.030932
Iter 70: Loss = 0.023577
Iter 80: Loss = 0.008938
Iter 90: Loss = 0.002542
Iter 100: Loss = 0.000785
Iter 110: Loss = 0.000445
Iter 120: Loss = 0.000238
Iter 130: Loss = 0.000157
Iter 140: Loss = 0.000071
Iter 150: Loss = 0.000032
Iter 160: Loss = 0.000018
Iter 170: Loss = 0.000002
Iter 180: Loss = 0.000001

--- 🔍 RÉSULTATS FINAUX ---
Modèle (Position) :
 [[-5.0003642e-01 -5.0007254e-01 -8.5165346e-01]
 [ 3.0319451e-04  5.0024307e-01  4.4802549e-01]
 [ 5.0012696e-01 -5.0005507e-01 -9.6425557e-01]]
Modèle (Couleur) : [[0.1798514  0.30954397 0.6690975 ]]
